# Investigate SNN activity

Load a trained `event_frame_snn` checkpoint and inspect per-layer spike / membrane
activity across the CNN-matched encoder–decoder (14 LIFs: `down1_a` … `up1_b`).
Metrics diagnose dead layers (high silent fraction) vs saturating ones
(high hyperactive fraction). Stage overview maps use each stage’s `_b` LIF.

Uses `BallHeatmapSNN.forward_with_activity` + `horus.eval.snn_activity`.

**Note:** retrain after the CNN-matched architecture change — old shallow
`event_frame_snn.pt` checkpoints will not load.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from horus.config import load_config, repo_root
from horus.data.dataset import build_dataloader
from horus.eval import ARCH_REGISTRY, load_model, summarize_activity
from horus.eval.metrics import input_channels
from horus.models import heatmap_to_coords
from horus.models.snn import LAYER_NAMES, STAGE_NAMES, BallHeatmapSNN

ARCH = "event_frame_snn"
SPLIT = "val"
BATCH_SIZE = 4
SAMPLE_IDX = 0  # which batch sample to visualize in detail
TOP_K_CHANNELS = 8

ROOT = repo_root()
cfg = load_config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_, modality = ARCH_REGISTRY[ARCH]
ckpt_path = ROOT / "checkpoints" / f"{ARCH}.pt"
if not ckpt_path.is_file():
    raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

model = load_model(ARCH, ckpt_path, device)
assert isinstance(model, BallHeatmapSNN)
ckpt_meta = torch.load(ckpt_path, map_location="cpu", weights_only=True)
print(
    f"arch={ARCH} modality={modality} in_ch={input_channels(modality, cfg)} "
    f"epoch={ckpt_meta.get('epoch')} val_px={ckpt_meta.get('val_px')} device={device}"
)

loader = build_dataloader(
    cfg, SPLIT, modality, batch_size=BATCH_SIZE, num_workers=0, shuffle=False
)
batch = next(iter(loader))
x = batch["input"].to(device)
with torch.no_grad():
    pred, trace = model.forward_with_activity(x)
    pred = pred.cpu()
gt, coords = batch["heatmap"], batch["coords"]
pred_xy = heatmap_to_coords(pred)
stats = summarize_activity(trace)


## Per-layer activity metrics

- **mean_membrane** — average pre-reset potential  
- **spike_frac** — fraction of neurons that fire at least once in the window  
- **rate** — spikes per neuron per timestep  
- **silent** — never fire (dead units)  
- **hyper** — fire on nearly every timestep (default threshold 0.9 → all 3 steps)


In [ ]:
print(f"{'layer':<14} {'mean_v':>8} {'spike%':>8} {'rate':>8} {'silent%':>8} {'hyper%':>8}")
print("-" * 58)
for name in LAYER_NAMES:
    m = stats[name]
    print(
        f"{name:<14} {m['mean_membrane']:8.4f} {100*m['spike_frac']:7.2f}% "
        f"{m['spikes_per_neuron_per_timestep']:8.4f} {100*m['silent_frac']:7.2f}% "
        f"{100*m['hyperactive_frac']:7.2f}%"
    )


## Layer representations

For one sample: event preview, mean spike-rate / membrane maps for each spatial
stage’s `_b` LIF, top-k channels for selected stages, then GT / pred overlay.


In [ ]:
def preview_event(chw: torch.Tensor) -> np.ndarray:
    """Last polarity pair as R=pos, B=neg."""
    x = chw.detach().float().cpu().numpy()
    _, h, w = x.shape
    pos, neg = x[-2], x[-1]
    scale = max(float(pos.max()), float(neg.max()), 1e-8)
    rgb = np.zeros((h, w, 3), dtype=np.float32)
    rgb[..., 0] = pos / scale
    rgb[..., 2] = neg / scale
    return np.clip(rgb, 0, 1)


def spatial_mean(t: torch.Tensor) -> np.ndarray:
    """(T, C, H, W) → mean over T and C → (H, W)."""
    return t.float().mean(dim=(0, 1)).cpu().numpy()


def top_channel_maps(spikes: torch.Tensor, k: int) -> tuple[np.ndarray, list[int]]:
    """Return (k, H, W) maps for the k channels with highest mean rate."""
    # spikes: (T, C, H, W)
    rates = spikes.float().mean(dim=(0, 2, 3))  # (C,)
    k = min(k, rates.numel())
    idx = torch.topk(rates, k).indices.tolist()
    maps = spikes[:, idx].float().mean(dim=0).cpu().numpy()  # (k, H, W)
    return maps, idx


# One overview column per spatial stage (use the block's second LIF).
STAGE_LIF = tuple(f"{s}_b" for s in STAGE_NAMES)

i = SAMPLE_IDX
frame = preview_event(batch["input"][i])
g = gt[i, 0].numpy()
p = pred[i, 0].numpy()
gx, gy = coords[i].tolist()
px, py = pred_xy[i].tolist()

n_stage = len(STAGE_LIF)
fig, axes = plt.subplots(2, n_stage + 1, figsize=(2.4 * (n_stage + 1), 5.5))
axes[0, 0].imshow(frame)
axes[0, 0].set_title(f"{batch['game'][i]}/{batch['clip'][i]} f={batch['frame'][i]}")

for col, name in enumerate(STAGE_LIF, start=1):
    layer = trace[name]
    rate_map = spatial_mean(layer.spikes[i])
    mem_map = spatial_mean(layer.membrane[i])
    im = axes[0, col].imshow(rate_map, cmap="magma")
    axes[0, col].set_title(f"{name}\nrate", fontsize=9)
    fig.colorbar(im, ax=axes[0, col], fraction=0.046)
    im = axes[1, col].imshow(mem_map, cmap="viridis")
    axes[1, col].set_title(f"{name}\nmem", fontsize=9)
    fig.colorbar(im, ax=axes[1, col], fraction=0.046)

axes[1, 0].imshow(frame)
axes[1, 0].imshow(p, cmap="hot", alpha=0.55, vmin=0, vmax=1)
axes[1, 0].scatter([gx], [gy], c="lime", s=40, marker="x", label="gt")
axes[1, 0].scatter([px], [py], c="cyan", s=40, marker="+", label="pred")
axes[1, 0].set_title("overlay")
axes[1, 0].legend(loc="upper right", fontsize=8)

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Top-k most active channels for a few key stages (_b LIF)
for name in ("down1_b", "bottleneck_b", "up1_b"):
    layer = trace[name]
    maps, idxs = top_channel_maps(layer.spikes[i], TOP_K_CHANNELS)
    k = maps.shape[0]
    fig, axes = plt.subplots(1, k, figsize=(2.2 * k, 2.4))
    if k == 1:
        axes = [axes]
    for ax, ch_map, ch in zip(axes, maps, idxs):
        ax.imshow(ch_map, cmap="magma")
        ax.set_title(f"ch {ch}", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"{name} — top-{k} channels by mean spike rate", fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
# Full batch: GT vs pred heatmaps (same layout as visualize_models)
n = pred.shape[0]
fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n))
if n == 1:
    axes = axes[None, :]

for j in range(n):
    fr = preview_event(batch["input"][j])
    gj, pj = gt[j, 0].numpy(), pred[j, 0].numpy()
    gxj, gyj = coords[j].tolist()
    pxj, pyj = pred_xy[j].tolist()

    axes[j, 0].imshow(fr)
    axes[j, 0].set_title(f"{batch['game'][j]}/{batch['clip'][j]} {batch['frame'][j]}")
    axes[j, 1].imshow(gj, cmap="hot", vmin=0, vmax=1)
    axes[j, 1].set_title("GT heatmap")
    axes[j, 2].imshow(pj, cmap="hot", vmin=0, vmax=1)
    axes[j, 2].set_title(f"pred  peak=({pxj:.0f},{pyj:.0f})")
    axes[j, 3].imshow(fr)
    axes[j, 3].imshow(pj, cmap="hot", alpha=0.55, vmin=0, vmax=1)
    axes[j, 3].scatter([gxj], [gyj], c="lime", s=40, marker="x", label="gt")
    axes[j, 3].scatter([pxj], [pyj], c="cyan", s=40, marker="+", label="pred")
    axes[j, 3].set_title(f"overlay ({ARCH})")
    if j == 0:
        axes[j, 3].legend(loc="upper right", fontsize=8)

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()
